In [1]:
import torch

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
# get unique chars that occur in the data
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [3]:
# Tokenization (we are building a character level language model so this is just mapping characters to integers)
# Define tokenizer
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder:

# Tokenize the data
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [4]:
# Split data into train and validation sets
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [5]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    


In [ ]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel (every forward and backward pass of the transformer)?
block_size = 8 # what is the maximum context length for predictions? (repeat from above)


def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # dims ?
    x = torch.stack([data[i:i+block_size] for i in ix]) # stack ? is this like append in numpy for tensors?
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train') # get batch of chunks
print("inputs:")
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

# ================================================
# For each batch, process each chunk in the batch in parallel

# So here is one design decision that parts ways from the MLP architecture:
# We dont not have a fixed context size (that is just block_size). This is purposeful (not just efficiency)
# This is so that the transformer is able to expect any context length (up to block_size). This means that during inference, we can sample with any context length we want (up to block_size)
# so here we define the "time dimension" of the transformer (how it reads in data)

# Now we define the "batch dimension" of the transformer (how it processes data in parallel)
# This is simply done for efficiency (GPUs are very good at parallel processing). Each chunk (batch) is processed totally independently
# (this is not the self-attention mechanism)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1] # note the variable context length (up to block_size)
        target = yb[b, t]
        #print(f'when b={b} and t={t}, the context is "{context.tolist()}" and the target is "{target}"')

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
